![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🔪 Top 10 Inpatient Surgeries — ETL Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-CIHI-1B3A6B?style=flat-square)

> **Analytical Question:** What are the top 10 surgeries causing hospitalization in NS vs. Canada?

This notebook extracts and cleans **CIHI Table 4 — Top 10 Inpatient Surgeries** from 8 annual CIHI Excel workbooks (2017–18 through 2024–25). It is the surgical complement to the MRDx pipeline — same architecture, different table and column names.

---

## 📦 Source Data

| Files | Content | Fiscal Years |
|-------|---------|-------------|
| `Hospitalization rates & others YYYY.xlsx` × 8 | Table 4: Top 10 inpatient surgical interventions by province | 2017–2018 to 2024–2025 |

**Source:** CIHI — Hospital Morbidity Database (HMDB)

---

## 🔧 ETL Architecture

Identical to `Hospitalization_Causes.ipynb` — same pipeline structure, different sheet detection keywords and column names.

```
discover_files()
    └── for each .xlsx:
          find_cihi_sheet()    ← detect Table 4 via 'inpatient surgeries' + 'surgical interventions'
          read_cihi_table()    ← dynamic header row detection
          process_file()       ← clean + tag fiscal year
    └── concat → sort → export
```

---

## ⚠️ Sheet Name Changes

| Fiscal Years | Sheet Name |
|-------------|-----------|
| 2017–2022 | `'4. Top 10 inp surgeries'` |
| 2023–2025 | `'Table 4'` |

Content-based detection handles this automatically.

---


## Step 1 — Imports & Config


In [ ]:
import pandas as pd
from pathlib import Path
import re

DATA_DIR    = Path(r"C:\data\capstone\Q4")
OUTPUT_DIR  = Path("clean")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "hospitalization_top10_inp_surgeries.csv"


## Step 2 — Content-Based Sheet Detection (Table 4)

Table 4 is identified by three keywords that appear together:
- `"inpatient surgeries"` — distinguishes from other tables
- `"surgical interventions"` — the column name for surgery type
- `"number of inpatient surgeries"` — the count column header

All three must be present to avoid false matches with other sheets.


In [ ]:
def find_cihi_sheet(xls):
    """
    Finds CIHI Table 4 (Top 10 inpatient surgeries).

    Uses three-keyword detection to avoid false matches.
    Handles sheet name changes across fiscal years automatically.
    """
    for sheet in xls.sheet_names:
        try:
            preview = pd.read_excel(xls, sheet_name=sheet, nrows=25)
            text    = " ".join(preview.astype(str).values.flatten()).lower()

            if (
                "inpatient surgeries"     in text
                and "surgical interventions" in text
                and "number of inpatient surgeries" in text
            ):
                return sheet

        except Exception:
            continue

    raise ValueError("CIHI Table 4 (Top 10 inpatient surgeries) not found in workbook")


## Step 3 — Dynamic Header Row Detection

Same approach as `Hospitalization_Causes.ipynb` — scan for the row containing `'Province'` and use it as the column header.


In [ ]:
def read_cihi_table(path):
    xls   = pd.ExcelFile(path)
    sheet = find_cihi_sheet(xls)
    print(f"   ✅ Sheet used: {sheet}")

    df = pd.read_excel(xls, sheet_name=sheet, header=None)

    header_row = df[
        df.apply(
            lambda r: r.astype(str).str.contains("province", case=False, na=False).any(),
            axis=1
        )
    ].index[0]

    df = df.iloc[header_row:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    return df


## Step 4 — Process One Workbook

Identical logic to `Hospitalization_Causes.ipynb` but with Table 4 column names:
- `Surgical interventions` (instead of `Most responsible diagnosis`)
- `Number of inpatient surgeries` (instead of `Number of inpatient hospitalizations`)
- `Percentage of inpatient surgeries`
- `Average acute length of stay of inpatient surgeries`


In [ ]:
def process_file(path, fiscal_year_range):
    print(f"   🧹 Processing {fiscal_year_range}")

    fiscal_year = int(fiscal_year_range.split("-")[1])

    df = read_cihi_table(path)

    # ── Remove junk columns ──────────────────────────────────
    df = df.loc[:, ~df.columns.astype(str).str.contains("unnamed", case=False)]

    # ── Keep first 6 columns (Table 4 layout) ───────────────
    df = df.iloc[:, :6]
    df.columns = [
        "Province/territory",
        "Rank",
        "Surgical interventions",
        "Number of inpatient surgeries",
        "Percentage of inpatient surgeries",
        "Average acute length of stay of inpatient surgeries",
    ]

    df = df.drop(columns=["Rank"])
    df = df[df["Province/territory"].notna()]

    # ── Strip footnote symbols (e.g. 'Ont.†' → 'Ont.') ──────
    df["Province/territory"] = (
        df["Province/territory"]
        .astype(str)
        .str.replace(r"[†‡*+§]+$", "", regex=True)
        .str.strip()
    )

    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna()
    df["Fiscal Year"] = fiscal_year

    return df


## Step 5 — Discover Files & Run Pipeline


In [ ]:
def discover_files():
    files = {}
    for f in DATA_DIR.glob("Hospitalization rates & others *.xlsx"):
        match = re.search(r"\d{4}-\d{4}", f.name)
        if match:
            files[match.group()] = f
    if not files:
        raise FileNotFoundError("No CIHI hospitalization files found")
    return dict(sorted(files.items()))


def run_pipeline():
    print("🚀 CIHI Table 4 – Top 10 Inpatient Surgeries ETL\n")

    files  = discover_files()
    frames = []

    for fy, path in files.items():
        frames.append(process_file(path, fy))

    final_df = pd.concat(frames, ignore_index=True)
    final_df = final_df.sort_values(
        ["Fiscal Year", "Province/territory", "Surgical interventions"]
    )

    final_df.to_csv(OUTPUT_FILE, index=False)

    print("\n✅ ETL Complete")
    print(f"📁 Output : {OUTPUT_FILE}")
    print(f"📊 Rows   : {len(final_df)}")

    return final_df


final_df = run_pipeline()


## ✅ Output Summary

| Output | Location | Rows | Fiscal Years |
|--------|----------|------|-------------|
| Clean CSV | `./clean/hospitalization_top10_inp_surgeries.csv` | ~831 | 2017–18 to 2024–25 |

**Final schema:** `Province/territory` \| `Surgical interventions` \| `Number of inpatient surgeries` \| `Percentage of inpatient surgeries` \| `Average acute length of stay of inpatient surgeries` \| `Fiscal Year`

**Feeds into:**
- 📊 Power BI — Top 10 Surgeries horizontal bar chart (NS vs. Canada)
- 📊 Power BI — Trend: how surgical volumes and LOS shift across 8 fiscal years

> ⚠️ **COVID note:** 2020–2021 surgical volumes were significantly reduced due to pandemic elective surgery cancellations. Flag this year in trend visuals.


---

*Nova Scotia Health & Population Analytics · CIHI Hospital Morbidity Database*
